In [1]:
from langchain_core.messages import RemoveMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")


def chat_node(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


def delete_old_messages(state: MessagesState):
    messages = state["messages"]
    if len(messages) > 10:
        to_delete = [RemoveMessage(id=m.id) for m in messages[:6]]
        return {"messages": to_delete}
    return {"messages": []}


builder = StateGraph(MessagesState)
builder.add_node("chat", chat_node)
builder.add_node("cleanup", delete_old_messages)
builder.add_edge(START, "chat")
builder.add_edge("chat", "cleanup")
builder.add_edge("cleanup", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer) 

In [2]:
config = {"configurable": {"thread_id": "1"}}


def show_state():
    msgs = graph.get_state(config).values["messages"]
    for m in msgs:
        print("-", type(m).__name__, ":", str(m.content)[:50])
    print("total messages:", len(msgs))

In [3]:
questions = [
    "Hi, I am Aayush",
    "Tell me about LangGraph",
    "What is a checkpointer?",
    "What is a thread id?",
    "Explain short-term memory",
    "Explain trimming",
    "What is RemoveMessage?",
]

for q in questions:
    graph.invoke({"messages": [HumanMessage(content=q)]}, config=config)

show_state()

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


- HumanMessage : What is a thread id?
- AIMessage : [{'type': 'text', 'text': 'A **`thread_id`** (Thre
- HumanMessage : Explain short-term memory
- AIMessage : [{'type': 'text', 'text': 'In AI and agentic syste
- HumanMessage : Explain trimming
- AIMessage : [{'type': 'text', 'text': 'In the context of LLMs 
- HumanMessage : What is RemoveMessage?
- AIMessage : [{'type': 'text', 'text': 'In LangGraph, **`Remove
total messages: 8
